# Train YOLO26n detection — Strawberry Vision Pi (Phase 2)

Trains the detection model on Zenodo strawberry imagery (record 6126677, 813 images, 3 classes: ripe / unripe / peduncle).

Two runs:
- **yolo26n** — production model that ships to the Pi (size constraint: must fit the NCNN ~67 ms/image budget).
- **yolo26s** — ablation upper-bound. Same data, same hyperparameters; the only variable is model size. Tells us the accuracy cost of the Pi-fit constraint.

**Runtime**: switch to `Runtime → Change runtime type → A100` (Colab Pro). T4 also works but A100 finishes both runs in ~25 min vs ~70 min on T4.

**Out of scope here**: NCNN export (do that on Mac after pulling `best.pt`); Hailo HEF conversion (Phase 5).

## 1. Environment

In [ ]:
!pip install -q ultralytics
import torch, ultralytics
print(f'ultralytics {ultralytics.__version__}')
print(f'torch       {torch.__version__}')
print(f'CUDA        {torch.cuda.is_available()} / {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## 2. Clone the repo

Public read-only HTTPS clone — no auth required.

In [ ]:
%cd /content
!rm -rf strawb-analysis
!git clone https://github.com/AKarode/strawb-analysis.git
%cd strawb-analysis

## 3. Download Zenodo dataset

Pulls record 6126677 (~1.5 GB), extracts the inner `strawberries.zip`, and places it at `data/zenodo/strawberries/{training,validation}/` — the layout `train_detect.py` expects.

In [ ]:
!mkdir -p data/zenodo/_dl
!curl -sSL -o data/zenodo/_dl/zenodo.zip 'https://zenodo.org/api/records/6126677/files-archive'
!unzip -q -o data/zenodo/_dl/zenodo.zip -d data/zenodo/_dl/
!ls data/zenodo/_dl/

In [ ]:
# The outer archive contains a nested strawberries.zip. Unpack it into
# data/zenodo/strawberries/. Defensive: handle both the nested-zip case and
# the case where the outer archive already contained the directories.
import shutil, subprocess, zipfile
from pathlib import Path

dl = Path('data/zenodo/_dl')
target = Path('data/zenodo/strawberries')
target.mkdir(parents=True, exist_ok=True)

candidates = [p for p in dl.glob('*.zip') if p.name != 'zenodo.zip']
if candidates:
    inner = candidates[0]
    print(f'extracting nested {inner.name}')
    with zipfile.ZipFile(inner) as zf:
        zf.extractall(dl)

for src_name in ('training', 'validation', 'strawberries.yaml', 'names.txt', 'data.yaml'):
    # Source may be either at dl/<name> or dl/strawberries/<name>
    for src in (dl / src_name, dl / 'strawberries' / src_name):
        if src.exists():
            dst = target / src_name
            if dst.exists():
                if dst.is_dir():
                    shutil.rmtree(dst)
                else:
                    dst.unlink()
            shutil.move(str(src), str(dst))
            break

shutil.rmtree(dl, ignore_errors=True)
print('---')
subprocess.run(['ls', '-la', str(target)])

In [ ]:
# Sanity-check counts. Expected: 654 train images, 159 val images.
!echo 'training jpgs:' $(find data/zenodo/strawberries/training -iname '*.jpg' | wc -l)
!echo 'training txts:' $(find data/zenodo/strawberries/training -iname '*.txt' | wc -l)
!echo 'val      jpgs:' $(find data/zenodo/strawberries/validation -iname '*.jpg' | wc -l)
!echo 'val      txts:' $(find data/zenodo/strawberries/validation -iname '*.txt' | wc -l)
!cat data/zenodo/strawberries/strawberries.yaml 2>/dev/null || cat data/zenodo/strawberries/data.yaml 2>/dev/null || echo 'no upstream yaml'

## 4. Mount Google Drive for checkpoint persistence

Colab runtimes disconnect after ~24 h on Pro (or earlier on idle). Writing run outputs to Drive means a disconnect doesn't lose epochs.

Skip this cell if you'd rather not mount Drive — just change `PROJECT_DIR` below to `runs/detect` and download `best.pt` manually before disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
PROJECT_DIR = '/content/drive/MyDrive/strawb-models/runs/detect'
pathlib.Path(PROJECT_DIR).mkdir(parents=True, exist_ok=True)
print(f'training output dir: {PROJECT_DIR}')

## 5a. Train production model — yolo26n

100 epochs, 640×640, batch 64 (good for A100 40GB). Tune down to 32 on T4 / V100, or 48 on L4. Early stopping at `--patience 20` typically ends the run between epoch 60–80.

In [ ]:
!python scripts/train_detect.py \
    --data-root data/zenodo/strawberries \
    --weights yolo26n.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 64 \
    --device 0 \
    --project "$PROJECT_DIR" \
    --name zenodo_yolo26n \
    --patience 20

## 5b. Train ablation upper-bound — yolo26s (optional)

**Same data, same hyperparameters as 5a — only the model size changes.** This run is the accuracy upper-bound: it tells us how much mAP we sacrifice by picking `n` for the Pi-fit constraint.

`yolo26s` is ~3× the params of `n`. Drop `--batch` to 48 if you're on a smaller GPU. Skip this cell entirely if you only want the production model.

In [ ]:
!python scripts/train_detect.py \
    --data-root data/zenodo/strawberries \
    --weights yolo26s.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 48 \
    --device 0 \
    --project "$PROJECT_DIR" \
    --name zenodo_yolo26s \
    --patience 20

## 6. Ablation comparison — n vs s

Pulls best-epoch validation metrics from each run's `results.csv` and prints a side-by-side table. Whichever runs you completed will appear; missing runs are flagged.

In [ ]:
import os
import pandas as pd

RUNS = [
    ('yolo26n (production)', f'{PROJECT_DIR}/zenodo_yolo26n'),
    ('yolo26s (ablation)',   f'{PROJECT_DIR}/zenodo_yolo26s'),
]

rows = []
for label, run_dir in RUNS:
    csv = os.path.join(run_dir, 'results.csv')
    if not os.path.exists(csv):
        rows.append({'model': label, 'status': 'not run'})
        continue
    df = pd.read_csv(csv)
    df.columns = [c.strip() for c in df.columns]
    best_idx = df['metrics/mAP50-95(B)'].idxmax()
    best = df.loc[best_idx]
    rows.append({
        'model':         label,
        'epochs':        int(df['epoch'].max()) + 1,
        'best_epoch':    int(best['epoch']),
        'mAP50-95':      round(float(best['metrics/mAP50-95(B)']), 4),
        'mAP50':         round(float(best['metrics/mAP50(B)']),    4),
        'precision':     round(float(best['metrics/precision(B)']), 4),
        'recall':        round(float(best['metrics/recall(B)']),    4),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

# Compute the cost of the n-vs-s tradeoff if both completed.
if len(summary) == 2 and 'mAP50-95' in summary.columns and summary['mAP50-95'].notna().all():
    delta = float(summary.iloc[1]['mAP50-95']) - float(summary.iloc[0]['mAP50-95'])
    print(f'\naccuracy cost of choosing n over s: {delta:+.4f} mAP50-95')
    print(f'(positive = s is better; negative = n surprisingly beats s on this dataset)')

## 7. Inspect plots

Loops over both runs and shows the standard Ultralytics outputs: training/val loss curves, confusion matrices, and a sample of validation predictions.

In [ ]:
import os
from IPython.display import Image, display, Markdown

RUN_NAMES = ['zenodo_yolo26n', 'zenodo_yolo26s']
PLOT_FILES = (
    'results.png',
    'PR_curve.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'val_batch0_pred.jpg',
)

for name in RUN_NAMES:
    run_dir = f'{PROJECT_DIR}/{name}'
    if not os.path.isdir(run_dir):
        continue
    display(Markdown(f'### {name}'))
    for fname in PLOT_FILES:
        path = os.path.join(run_dir, fname)
        if os.path.exists(path):
            display(Markdown(f'**{fname}**'))
            display(Image(path))

## 8. Download weights

Both `best.pt` files are already on Drive at `MyDrive/strawb-models/runs/detect/<run>/weights/best.pt`. Sync via the Drive desktop client, or trigger browser downloads below.

In [ ]:
import os
from google.colab import files

for name in ('zenodo_yolo26n', 'zenodo_yolo26s'):
    best = f'{PROJECT_DIR}/{name}/weights/best.pt'
    if os.path.exists(best):
        print(f'downloading {name}/best.pt')
        files.download(best)
    else:
        print(f'(skip) {name}/best.pt does not exist')

## Next steps (off-Colab)

1. Mac side: `mkdir -p models/detect && cp ~/Downloads/best.pt models/detect/yolo26n_zenodo.pt` (and `yolo26s_zenodo.pt` if you ran the ablation).
2. Mac side: `python scripts/smoke_test_yolo26.py` (or a dedicated `scripts/export_detect_ncnn.py` once authored) to produce `models/detect/yolo26n_zenodo_ncnn_model/`.
3. Pi side: pull the NCNN model down and run the CPU benchmark — Phase 4 territory.
4. Document the n-vs-s accuracy delta from section 6 in the project notes — that's the defensible answer to "why didn't you use the bigger model?"